In [ ]:
# Cell 1 — Run configuration
import hashlib, json, time
from pathlib import Path

PUBLICATION_SEEDS=[42,1337,2026,31415,271828]
RUN_SEED=42  # CHANGE ONLY THIS

# Optional exact completed same-seed plain-Gated parent path/ZIP.
# Leave empty for automatic /kaggle/input discovery.
GATED_SOURCE_OVERRIDE=""

PARENT_GATED_PROTOCOL_SHA="627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
FACETGATE_COMMON_SHA256="735b3e08413949cdb148fe4ce570493b5cd410e562dad7a12748b2e5f2fd6a38"
SHARED_FORWARD_PROTOCOL_SHA256="37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
TASK_INSTRUCTION="Encode this meme for six independent annotations: offensive, vulgar, abuse, sarcastic, language (Bangla/English/Banglish), and zero or more of seven real target groups. Do not impose logical implications between tasks. "
TASK_INSTRUCTION_SHA=hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest()

if RUN_SEED not in PUBLICATION_SEEDS:
    raise ValueError(f"RUN_SEED must be one of {PUBLICATION_SEEDS}, got {RUN_SEED}")
if TASK_INSTRUCTION_SHA!="bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f":
    raise RuntimeError("Canonical task instruction drift.")

print({
    "model":"meta-llama/Llama-3.2-11B-Vision-Instruct",
    "seed":RUN_SEED,
    "gated_protocol":PARENT_GATED_PROTOCOL_SHA,
    "facetgate_protocol":FACETGATE_COMMON_SHA256,
    "shared_forward_protocol":SHARED_FORWARD_PROTOCOL_SHA256,
    "TEST_forwards":0,
},flush=True)


In [ ]:
# Cell 2 — Resolve a completed same-seed Llama Gated parent before any training
# Safe to run after Cell 1 by itself. It performs metadata/ZIP-directory inspection only:
# no model download, no dataset forward, no TEST forward.
import os, json, hashlib, zipfile, shutil, re
from pathlib import Path

INPUT_ROOT=Path("/kaggle/input")
WORKING_ROOT=Path("/kaggle/working")
GATED_EXPECTED_MODEL_ID="meta-llama/Llama-3.2-11B-Vision-Instruct"
GATED_EXPECTED_REVISION="9eb2daaa8597bf192a8b0e73f848f3a102794df5"
GATED_CANDIDATE_STEPS={200,400,600,800,932}

def _read_json_file(p):
    try: return json.loads(Path(p).read_text(encoding="utf-8"))
    except Exception: return None

def _gated_run_config_ok(cfg):
    if not isinstance(cfg,dict): return False
    return (
        int(cfg.get("seed",-1))==int(RUN_SEED)
        and cfg.get("model_id")==GATED_EXPECTED_MODEL_ID
        and cfg.get("model_revision")==GATED_EXPECTED_REVISION
        and cfg.get("frozen_common_gated_protocol_sha256")==PARENT_GATED_PROTOCOL_SHA
        and cfg.get("task_instruction_sha256")==TASK_INSTRUCTION_SHA
        and cfg.get("allow_final_test") is False
        # A prior Shared-Forward FacetGate run is NOT reclassified as a
        # plain-Gated parent; its specialist trajectory must remain bound to
        # that same joint run and is resumed by the simultaneous child itself.
        and cfg.get("execution_mode")!="SIMULTANEOUS_FROM_SCRATCH"
        and not cfg.get("facetgate_common_sha256")
    )

def _checkpoint_member_set_ok(names,prefix,step):
    cp=f"{prefix}checkpoints/checkpoint-{int(step):04d}/"
    req=[
        cp+"classifier_heads.pt",
        cp+"adapter/adapter_config.json",
    ]
    if not all(x in names for x in req): return False
    return (cp+"adapter/adapter_model.safetensors" in names or cp+"adapter/adapter_model.bin" in names)

def _inspect_direct_root(root):
    root=Path(root)
    fs=_read_json_file(root/"final_status.json")
    cfg=_read_json_file(root/"run_config.json")
    if not isinstance(fs,dict) or not _gated_run_config_ok(cfg): return None
    if int(fs.get("seed",-1))!=int(RUN_SEED): return None
    test_rows=int(fs.get("test_rows_forwarded",0) or 0)
    if test_rows!=0:
        return {"state":"REJECTED_TEST_TOUCHED","kind":"directory","source":str(root),"final_status":fs}
    complete=bool(fs.get("training_complete")) and bool(fs.get("validation_complete"))
    if complete:
        step=int(fs.get("best_step",-1))
        cp=root/"checkpoints"/f"checkpoint-{step:04d}"
        thresholds=root/"artifacts"/"thresholds.json"
        valid_cp=(
            step in GATED_CANDIDATE_STEPS and
            (cp/"classifier_heads.pt").is_file() and
            (cp/"adapter"/"adapter_config.json").is_file() and
            any((cp/"adapter"/n).is_file() for n in ["adapter_model.safetensors","adapter_model.bin"]) and
            thresholds.is_file()
        )
        if not valid_cp:
            return {
                "state":"COMPLETE_STATUS_BUT_ARTIFACTS_MISSING","kind":"directory","source":str(root),
                "best_step":step,"final_status":fs,"run_config":cfg
            }
        return {
            "state":"COMPLETE_GATED_FOUND","kind":"directory","source":str(root),"root":str(root),
            "best_step":step,"checkpoint":str(cp),"thresholds":str(thresholds),
            "validation_freeze_id":fs.get("validation_freeze_id"),
            "final_status":fs,"run_config":cfg,
        }
    return {
        "state":"INCOMPLETE_RESUMABLE_GATED_FOUND","kind":"directory","source":str(root),"root":str(root),
        "global_step":int(fs.get("global_step",0) or 0),"stop_reason":fs.get("stop_reason"),
        "final_status":fs,"run_config":cfg,
    }

def _zip_json(z,name):
    try:
        with z.open(name) as f: return json.loads(f.read().decode("utf-8"))
    except Exception: return None

def _inspect_zip(zp):
    zp=Path(zp)
    try:
        with zipfile.ZipFile(zp) as z:
            names=set(z.namelist())
            status_members=[n for n in names if n.endswith("final_status.json")]
            reports=[]
            for sm in status_members:
                prefix=sm[:-len("final_status.json")]
                fs=_zip_json(z,sm)
                cfg=_zip_json(z,prefix+"run_config.json")
                if not isinstance(fs,dict) or not _gated_run_config_ok(cfg): continue
                if int(fs.get("seed",-1))!=int(RUN_SEED): continue
                test_rows=int(fs.get("test_rows_forwarded",0) or 0)
                if test_rows!=0:
                    reports.append({"state":"REJECTED_TEST_TOUCHED","kind":"zip","source":str(zp),"prefix":prefix,"final_status":fs})
                    continue
                complete=bool(fs.get("training_complete")) and bool(fs.get("validation_complete"))
                if complete:
                    step=int(fs.get("best_step",-1))
                    th=prefix+"artifacts/thresholds.json"
                    if step in GATED_CANDIDATE_STEPS and th in names and _checkpoint_member_set_ok(names,prefix,step):
                        reports.append({
                            "state":"COMPLETE_GATED_FOUND","kind":"zip","source":str(zp),"prefix":prefix,
                            "best_step":step,"validation_freeze_id":fs.get("validation_freeze_id"),
                            "final_status":fs,"run_config":cfg,
                        })
                    else:
                        reports.append({
                            "state":"COMPLETE_STATUS_BUT_ARTIFACTS_MISSING","kind":"zip","source":str(zp),"prefix":prefix,
                            "best_step":step,"final_status":fs,"run_config":cfg,
                        })
                else:
                    # Exact R4 safeguard ZIPs include the latest checkpoint and are the normal continuation source.
                    reports.append({
                        "state":"INCOMPLETE_RESUMABLE_GATED_FOUND","kind":"zip","source":str(zp),"prefix":prefix,
                        "global_step":int(fs.get("global_step",0) or 0),"stop_reason":fs.get("stop_reason"),
                        "final_status":fs,"run_config":cfg,
                    })
            if not reports: return None
            order={"COMPLETE_GATED_FOUND":0,"INCOMPLETE_RESUMABLE_GATED_FOUND":1,
                   "COMPLETE_STATUS_BUT_ARTIFACTS_MISSING":2,"REJECTED_TEST_TOUCHED":3}
            return sorted(reports,key=lambda x:(order.get(x["state"],9),len(x.get("prefix",""))))[0]
    except (zipfile.BadZipFile,OSError,PermissionError):
        return None

def _candidate_roots(search_working=False):
    roots=[]
    bases=[INPUT_ROOT]
    if search_working: bases.append(WORKING_ROOT)
    for base in bases:
        try:
            roots += [p.parent for p in base.rglob("final_status.json")]
        except Exception:
            pass
    return roots

def _candidate_zips(search_working=False):
    zips=[]
    bases=[INPUT_ROOT]
    if search_working: bases.append(WORKING_ROOT)
    for base in bases:
        try:
            for p in base.rglob("*.zip"):
                low=p.name.casefold()
                # Prefer relevant Llama Gated packages; an explicit override bypasses this filter.
                if ("llama" in low and "gated" in low) or f"seed{RUN_SEED}" in low:
                    zips.append(p)
        except Exception:
            pass
    return zips

def _refresh_gated_discovery(search_working=False):
    reports=[]
    override=str(GATED_SOURCE_OVERRIDE or "").strip()
    if override:
        p=Path(override)
        if not p.exists(): raise FileNotFoundError(f"GATED_SOURCE_OVERRIDE does not exist: {p}")
        if p.is_file() and zipfile.is_zipfile(p):
            r=_inspect_zip(p)
        else:
            root=p
            if p.is_file(): root=p.parent
            # If a checkpoint or nested artifact was supplied, walk upward to the run root.
            r=None
            for q in [root,*list(root.parents)[:7]]:
                rr=_inspect_direct_root(q)
                if rr is not None:
                    r=rr; break
        if r is None:
            raise RuntimeError("GATED_SOURCE_OVERRIDE exists but is not a compatible same-seed Llama Gated run.")
        reports=[r]
    else:
        seen=set()
        for root in _candidate_roots(search_working=search_working):
            key=("d",str(root))
            if key in seen: continue
            seen.add(key)
            r=_inspect_direct_root(root)
            if r is not None: reports.append(r)
        for zp in _candidate_zips(search_working=search_working):
            key=("z",str(zp))
            if key in seen: continue
            seen.add(key)
            r=_inspect_zip(zp)
            if r is not None: reports.append(r)

    complete=[x for x in reports if x.get("state")=="COMPLETE_GATED_FOUND"]
    if complete:
        # Multiple copies of the same validation freeze are fine; different freezes are not.
        freeze_ids={str(x.get("validation_freeze_id")) for x in complete}
        freeze_ids.discard("None")
        if len(freeze_ids)>1 and not override:
            raise RuntimeError(
                f"Multiple different completed same-seed Llama Gated validation freezes found: {sorted(freeze_ids)}. "
                "Set GATED_SOURCE_OVERRIDE to the intended parent."
            )
        # Direct mounted output > ZIP; shortest path/prefix breaks duplicate-copy ties deterministically.
        complete=sorted(complete,key=lambda x:(0 if x["kind"]=="directory" else 1,
                                                len(x.get("source",""))+len(x.get("prefix","")),
                                                x.get("source","")))
        return complete[0]

    incomplete=[x for x in reports if x.get("state")=="INCOMPLETE_RESUMABLE_GATED_FOUND"]
    if incomplete:
        return sorted(incomplete,key=lambda x:(-int(x.get("global_step",0)),
                                               0 if x["kind"]=="directory" else 1,
                                               x.get("source","")))[0]

    rejected=[x for x in reports if x.get("state") in {"COMPLETE_STATUS_BUT_ARTIFACTS_MISSING","REJECTED_TEST_TOUCHED"}]
    if rejected:
        return rejected[0]
    return {"state":"NO_GATED_FOUND","kind":None,"source":None,"seed":int(RUN_SEED)}

GATED_DISCOVERY=_refresh_gated_discovery(search_working=False)
print("\n"+"="*96)
print(" LLAMA GATED SAME-SEED TEST CELL ")
print("="*96)
print(json.dumps({
    "RUN_SEED":RUN_SEED,
    "state":GATED_DISCOVERY.get("state"),
    "kind":GATED_DISCOVERY.get("kind"),
    "source":GATED_DISCOVERY.get("source"),
    "best_step":GATED_DISCOVERY.get("best_step"),
    "global_step":GATED_DISCOVERY.get("global_step"),
    "validation_freeze_id":GATED_DISCOVERY.get("validation_freeze_id"),
    "decision":(
        "REUSE_EXISTING_GATED -> run only FacetGate/SHTE Target specialist"
        if GATED_DISCOVERY.get("state")=="COMPLETE_GATED_FOUND"
        else "SIMULTANEOUS_FROM_SCRATCH -> Gated + detached SHTE/FacetGate in the same shared-forward stream"
        if GATED_DISCOVERY.get("state") in {"NO_GATED_FOUND","INCOMPLETE_RESUMABLE_GATED_FOUND"}
        else "STOP: attached Gated candidate is not publication-valid"
    ),
    "TEST_forwards":0,
},indent=2),flush=True)
print("="*96+"\n")


In [ ]:
# CELL 3 — EMBEDDED VERIFIED EXECUTION PATHS
# Both children are self-contained and selected automatically from Cell 2.
import base64, zlib, hashlib

TARGET_ONLY_CHILD_ZLIB_B64='...'
TARGET_ONLY_CHILD_SHA256="9164386b07895d1035b9f268d356a9f6e80e085c6b5415db3ec1bc96d7e9676c"
SIMULTANEOUS_CHILD_ZLIB_B64='...'
SIMULTANEOUS_CHILD_SHA256="cc4141d69c7436b328e9593fd502bfba707d4d3711cf37738aff6bb9c6b7b331"

print({
    "target_only_child_sha256":TARGET_ONLY_CHILD_SHA256,
    "simultaneous_child_sha256":SIMULTANEOUS_CHILD_SHA256,
    "target_only_raw_bytes":159754,
    "simultaneous_raw_bytes":376144,
    "TEST_forwards":0,
},flush=True)


In [ ]:
# Cell 4 — Select, materialize, seed, and statically audit the execution branch
import base64, zlib, hashlib, json, ast, shutil
from pathlib import Path
import nbformat

WORK_ROOT=Path("/kaggle/working")/"M3Offend_Llama32_11B_Vision_FacetGate_AUTO_SEED_v1_0"
WORK_ROOT.mkdir(parents=True,exist_ok=True)

state=GATED_DISCOVERY.get("state")
if state=="COMPLETE_GATED_FOUND":
    EXECUTION_MODE="REUSE_EXISTING_GATED"
    _b64=TARGET_ONLY_CHILD_ZLIB_B64
    _sha=TARGET_ONLY_CHILD_SHA256
elif state in {"NO_GATED_FOUND","INCOMPLETE_RESUMABLE_GATED_FOUND"}:
    EXECUTION_MODE="SIMULTANEOUS_FROM_SCRATCH"
    _b64=SIMULTANEOUS_CHILD_ZLIB_B64
    _sha=SIMULTANEOUS_CHILD_SHA256
elif state=="REJECTED_TEST_TOUCHED":
    raise RuntimeError("Candidate Gated parent touched TEST; refusing development reuse.")
elif state=="COMPLETE_STATUS_BUT_ARTIFACTS_MISSING":
    raise RuntimeError(
        "A same-seed Gated run claims completion but required frozen checkpoint/threshold artifacts are missing. "
        "Do not silently replace this ambiguous parent; fix/remove it or set GATED_SOURCE_OVERRIDE."
    )
else:
    raise RuntimeError(f"Unhandled Gated discovery state: {state}")

raw=zlib.decompress(base64.b64decode(_b64))
if hashlib.sha256(raw).hexdigest()!=_sha:
    raise RuntimeError("Embedded child SHA-256 mismatch.")
text=raw.decode("utf-8")
if text.count("__RUN_SEED__")<1:
    raise RuntimeError("Embedded child has no controller seed placeholder.")
text=text.replace("__RUN_SEED__",str(int(RUN_SEED)))
nb=nbformat.reads(text,as_version=4)

# Static compile every code cell before any expensive child execution.
compile_errors=[]
for i,c in enumerate(nb.cells):
    if c.cell_type=="code":
        try:
            compile(c.source,f"child_cell_{i}","exec")
        except Exception as exc:
            compile_errors.append((i,repr(exc)))
if compile_errors:
    raise RuntimeError(f"Materialized child compile failures: {compile_errors}")

all_src="\n".join(c.source for c in nb.cells if c.cell_type=="code")
if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    required=[
        "REUSE_EXISTING_GATED",
        "TARGET_ONLY child requires a completed same-seed Gated parent",
        "FACETGATE_COMMON_SHA256",
        "ALLOW_LOCKED_TEST=False",
    ]
    forbidden=[
        "action\":\"RUN_OR_RESUME_EXACT_LLAMA_FROZEN_COMMON_GATED_R4_4_2_THEN_CONTINUE",
        "SIMULTANEOUS_FROM_SCRATCH",
    ]
    missing=[x for x in required if x not in all_src]
    bad=[x for x in forbidden if x in all_src]
    if missing or bad:
        raise RuntimeError(f"TARGET_ONLY semantic QA failed missing={missing} forbidden={bad}")
else:
    required=[
        "SIMULTANEOUS_FROM_SCRATCH",
        "SHARED-FORWARD FACETGATE/SHTE UPDATE",
        "facetgate_state",
        "save_facet_candidate",
        "facetgate_compatibility_replay",
        "dual_checkpoint_final_manifest",
        "shared_backbone_forward",
    ]
    missing=[x for x in required if x not in all_src]
    if missing:
        raise RuntimeError(f"SIMULTANEOUS semantic QA failed missing={missing}")
    # The fallback must not encode the old sequential scientific route.
    if "RUN_OR_RESUME_EXACT_LLAMA_FROZEN_COMMON_GATED_R4_4_2_THEN_CONTINUE" in all_src:
        raise RuntimeError("Sequential Gated->SHTE fallback leaked into simultaneous child.")

CHILD_NOTEBOOK=WORK_ROOT/f"Llama_FacetGate_seed{RUN_SEED}_{EXECUTION_MODE}.ipynb"
nbformat.write(nb,CHILD_NOTEBOOK)
print("[CHILD MATERIALIZED]",{
    "seed":RUN_SEED,
    "execution_mode":EXECUTION_MODE,
    "gated_discovery_state":state,
    "child":str(CHILD_NOTEBOOK),
    "code_cells":sum(c.cell_type=="code" for c in nb.cells),
    "compile_errors":0,
    "TEST_forwards":0,
},flush=True)

if state=="INCOMPLETE_RESUMABLE_GATED_FOUND":
    print(
        "[IMPORTANT] The discovered incomplete artifact is a plain-Gated run and is NOT a valid "
        "Shared-Forward optimizer-state resume because it has no synchronized SHTE history. "
        "The simultaneous child will start/resume only a provenance-compatible Shared-Forward run.",
        flush=True,
    )


In [ ]:
# CELL 5 — EXECUTE THE AUTOMATICALLY SELECTED PIPELINE IN A FRESH KERNEL
# The child owns the package/runtime/GPU lifecycle; this controller remains lightweight.

import subprocess, sys, time
from pathlib import Path

EXECUTED_NOTEBOOK = CHILD_NOTEBOOK.with_name(
    CHILD_NOTEBOOK.stem + "_EXECUTED.ipynb"
)

cmd = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    "--ExecutePreprocessor.timeout=-1",
    "--output",
    EXECUTED_NOTEBOOK.name,
    str(CHILD_NOTEBOOK),
]

print("[EXECUTING]", " ".join(cmd), flush=True)
started = time.time()
proc = subprocess.run(cmd, cwd=str(WORK_ROOT))
elapsed = time.time() - started

if proc.returncode != 0:
    raise RuntimeError(
        f"Selected Llama-3.2-11B-Vision FacetGate pipeline failed: "
        f"mode={EXECUTION_MODE} returncode={proc.returncode}. "
        f"Inspect {EXECUTED_NOTEBOOK if EXECUTED_NOTEBOOK.exists() else CHILD_NOTEBOOK}"
    )

print("[PIPELINE PROCESS EXITED CLEANLY]", {
    "seed": RUN_SEED,
    "execution_mode": EXECUTION_MODE,
    "elapsed_seconds": elapsed,
    "executed_notebook": str(EXECUTED_NOTEBOOK),
    "TEST_forwards": 0,
})


In [ ]:
# CELL 6 — FINAL CONTROLLER SUMMARY
import json
from pathlib import Path

print("="*100)
print("M3Offend Llama-3.2-11B-Vision FacetGate AUTO-SEED STATUS")
print("="*100)
print("Seed :",RUN_SEED)
print("Mode :",EXECUTION_MODE)
print("Initial plain-Gated discovery :",GATED_DISCOVERY.get("state"))
print("Executed child :",EXECUTED_NOTEBOOK)
if EXECUTION_MODE=="REUSE_EXISTING_GATED":
    print("Route : completed same-seed Gated frozen/reused -> SHTE/FacetGate Target-only")
else:
    print("Route : Gated + detached SHTE/FacetGate simultaneously in one Shared-Forward 932-step stream")

interesting=[]
for name in [
    "facetgate_result.json",
    "dual_checkpoint_final_manifest.json",
    "BASE_GATED_BINDING.json",
    "facetgate_target_thresholds.json",
    "paper_metrics_summary.csv",
]:
    for p in Path("/kaggle/working").rglob(name):
        s=str(p).casefold()
        if "llama" in s and ("facetgate" in s or "facegate" in s):
            interesting.append(p)

if interesting:
    print("\nResult artifacts:")
    for p in sorted(set(interesting)):
        print(" -",p)
        if p.name in {"facetgate_result.json","dual_checkpoint_final_manifest.json"}:
            try: print(json.dumps(json.loads(p.read_text(encoding="utf-8")),indent=2)[:6000])
            except Exception: pass
else:
    print("\nNo final FacetGate result manifest found yet. Inspect the executed child/live logs; "
          "a time-safeguarded Shared-Forward run may require the same-seed resume.")

print("\nDevelopment locked TEST forwards expected : 0")
print("="*100)
